# 🚀 Hunyuan3D-2.1 Cloud Server for Unity Editor Bridge

Run the official **Hunyuan3D-2.1** inference server on a **free Google Colab GPU (T4 / A100 / L4)** and connect it directly to your Unity Editor project.

### 📌 How to use:
1. In the Colab menu, go to **Runtime** ➔ **Change runtime type** ➔ Select **T4 GPU** (free) or better.
2. Run all cells below (or press **Ctrl + F9**).
3. Wait for the final cell to display your **Public Cloudflare Tunnel URL** (e.g. `https://xxxx.trycloudflare.com`).
4. Copy and paste that URL into the **Server URL** field inside Unity (**Tools ➔ Hunyuan3D ➔ Generator**).
5. Click **Check Server** in Unity ➔ Ready to generate 3D models from any Mac, laptop, or low-end PC!

In [ ]:
# Step 1: Check GPU availability
!nvidia-smi

In [ ]:
# Step 2: Clone official Tencent Hunyuan3D-2 repository
import os
if not os.path.exists('Hunyuan3D-2'):
    !git clone https://github.com/Tencent/Hunyuan3D-2.1.git Hunyuan3D-2 || git clone https://github.com/Tencent/Hunyuan3D-2.git Hunyuan3D-2
%cd Hunyuan3D-2

In [ ]:
# Step 3: Install dependencies for shape generation
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install fastapi uvicorn pydantic trimesh rembg
# Install cloudflared to create a free public HTTPS tunnel without account/token
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# Step 4: Apply Unity Bridge step-limit and shape-only optimization patch
patch_code = '''
import os, re
if os.path.exists('api_models.py'):
    with open('api_models.py', 'r') as f: content = f.read()
    content = re.sub(r'le=20', 'le=50', content)
    with open('api_models.py', 'w') as f: f.write(content)
    print('Patched api_models.py (inference steps allowed up to 50)')

if os.path.exists('model_worker.py'):
    with open('model_worker.py', 'r') as f: content = f.read()
    # wrap texture pipeline in try-except for shape-only stability
    if 'except Exception as e:' not in content and 'self.paint_pipeline = Hunyuan3DPaintPipeline(conf)' in content:
        content = content.replace('self.paint_pipeline = Hunyuan3DPaintPipeline(conf)',
            'try:\n            self.paint_pipeline = Hunyuan3DPaintPipeline(conf)\n        except:\n            self.paint_pipeline = None')
    with open('model_worker.py', 'w') as f: f.write(content)
    print('Patched model_worker.py')
'''
exec(patch_code)

In [ ]:
# Step 5: Start Hunyuan3D FastAPI server and Cloudflare Tunnel
import subprocess, time, re

print('Starting local FastAPI server on port 8081...')
server_process = subprocess.Popen([
    'python', 'api_server.py',
    '--host', '127.0.0.1',
    '--port', '8081'
])

# Start Cloudflare Tunnel
print('Starting Cloudflare Tunnel to expose server to Unity...')
tunnel_process = subprocess.Popen([
    'cloudflared', 'tunnel',
    '--url', 'http://127.0.0.1:8081'
], stderr=subprocess.PIPE, universal_newlines=True)

public_url = None
for line in tunnel_process.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print('\n' + '='*60)
print('🎉 HUNYUAN3D CLOUD SERVER IS READY FOR UNITY!')
print('='*60)
print(f'👉 Copy this Server URL into Unity: {public_url}')
print('='*60 + '\n')

# Keep running
server_process.wait()